#Problem Statement 3: Bike Demand Forecasting using Subagging vs Bagging vs Boosting (Regression)
Tanisha Sharma|23102C0077

Syllabus mapping: Bagging, Subagging, Random Forest, Boosting, K-fold CV
Objective: Predict hourly bike rental count and compare ensemble methods for regression.

Dataset (link):

UCI – Bike Sharing Dataset: https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset  

Problem Statement

A smart-city mobility team wants to forecast bike demand to improve fleet distribution and reduce shortages. Build ensemble regression models and compare performance using k-fold cross validation.

Expected Input

hour.csv (from the UCI dataset)
Example:
python task3_bike_regression_ensembles.py --data hour.csv --target cnt --kfold 5

Expected Output

Cross-validation metrics (mean ± std):
RMSE, MAE
Model comparison across:
Bagging: RandomForestRegressor
Subagging: BaggingRegressor with max_samples < 1.0
Boosting: GradientBoostingRegressor / XGBoostRegressor
Files:
cv_regression_results.csv
final_predictions.csv (ActualCnt, PredictedCnt)
Report section (short):
which model generalizes best and why (bias–variance intuition)
Mandatory Tasks / Deliverables

Perform K-Fold CV (k=5) (not just one split)
Use at least 2 ensemble hyperparameters and explain impact:
RF: n_estimators, max_depth
Subagging: max_samples, n_estimators
Boosting: learning_rate, n_estimators
Provide feature importance (tree-based) and list top 8 features


In [1]:
!pip install xgboost


In [2]:
from google.colab import files
uploaded = files.upload()   # Upload hour.csv


Saving bike+sharing+dataset.zip to bike+sharing+dataset.zip


In [4]:

# Bike Demand Forecasting - Ensemble Regression
# Bagging vs Subagging vs Boosting


import pandas as pd
import numpy as np

from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor, BaggingRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

import matplotlib.pyplot as plt


# 1. Load Dataset
data = pd.read_csv("hour.csv")

# Drop non-useful columns
data = data.drop(columns=["instant","dteday","casual","registered"], errors='ignore')

target = "cnt"
X = data.drop(columns=[target])
y = data[target]

print("Dataset shape:", X.shape)


# 2. K-Fold Setup

k = 5
kf = KFold(n_splits=k, shuffle=True, random_state=42)

results = []


# 3. Define Models


models = {

    "RandomForest_Bagging": RandomForestRegressor(
        n_estimators=150,
        max_depth=12,
        random_state=42
    ),

    "Subagging_BaggingRegressor": BaggingRegressor(
        estimator=DecisionTreeRegressor(),
        n_estimators=150,
        max_samples=0.7,
        random_state=42
    ),

    "GradientBoosting": GradientBoostingRegressor(
        n_estimators=150,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    ),

    "XGBoost": XGBRegressor(
        n_estimators=150,
        learning_rate=0.1,
        max_depth=5,
        objective="reg:squarederror",
        random_state=42
    )
}

predictions_store = {}


# 4. Cross Validation

for name, model in models.items():

    preds = cross_val_predict(model, X, y, cv=kf)

    rmse = np.sqrt(mean_squared_error(y, preds))
    mae = mean_absolute_error(y, preds)

    results.append([name, rmse, mae])
    predictions_store[name] = preds

    print(f"{name} -> RMSE: {rmse:.2f} , MAE: {mae:.2f}")

# Save CV results
cv_df = pd.DataFrame(results, columns=["Model","RMSE","MAE"])
cv_df.to_csv("cv_regression_results.csv", index=False)
print("\nSaved: cv_regression_results.csv")


# 5. Train Best Model Fully

best_model_name = cv_df.sort_values("RMSE").iloc[0]["Model"]
best_model = models[best_model_name]

best_model.fit(X, y)
final_preds = best_model.predict(X)

final_df = pd.DataFrame({
    "ActualCnt": y,
    "PredictedCnt": final_preds
})

final_df.to_csv("final_predictions.csv", index=False)
print("Saved: final_predictions.csv")

print("\nBest Model:", best_model_name)


# 6. Feature Importance

if hasattr(best_model, "feature_importances_"):
    importances = pd.Series(best_model.feature_importances_, index=X.columns)
    top_features = importances.sort_values(ascending=False).head(8)

    print("\nTop 8 Important Features:")
    print(top_features)

    plt.figure(figsize=(8,5))
    top_features.plot(kind="barh")
    plt.title("Top 8 Feature Importances")
    plt.show()


Dataset shape: (17379, 12)
RandomForest_Bagging -> RMSE: 45.35 , MAE: 27.52
Subagging_BaggingRegressor -> RMSE: 42.34 , MAE: 25.44
GradientBoosting -> RMSE: 63.17 , MAE: 43.22
XGBoost -> RMSE: 44.12 , MAE: 28.05

Saved: cv_regression_results.csv
Saved: final_predictions.csv

Best Model: Subagging_BaggingRegressor
